# Week 2, Lab 2: Context Window Management - How Your Twin Manages Its Working Memory

## The map before the territory

A model can only look at a limited amount of text on each call. That limit is called its **context window**. It is measured in *tokens*, which are small pieces of text, a bit shorter than a word on average. Everything you send has to fit: the system message, the whole chat history, any document text and the question. The model's own reply needs room too.

In week 1 you kept a `messages` list and sent all of it every time. That works for short chats. A long chat will not fit, so something has to go. This lab builds the part that decides what stays and what goes. It keeps the system message, keeps the newest turns and drops the oldest ones. There is no API call. The Reading tab explains counting size, looping over a list from the end, and building a new list from scratch.

## Exercise

### Goal
Write a function that shrinks a chat history until it fits in a token budget (the most tokens you allow yourself to send). The system message always stays. After that, keep the newest messages that fit and drop the oldest.

`count_tokens` is already written (about 4 characters make 1 token, never less than 1). You only write `trim_messages`. Hover over the code in the cell below for the steps and examples.

### What the asserts below test
The cell below the function builds a history of four messages and checks:
- a big budget keeps all four messages;
- a small budget keeps the system message and only the newest turn;
- the original list still has four messages afterwards;
- with a budget of 1, only the system message is left.

If all four hold, you see `All checks passed.`.

In [ ]:
def count_tokens(text: str) -> int:
    return len(text) // 4 + 1


def trim_messages(messages: list[dict], max_tokens: int) -> list[dict]:
    raise NotImplementedError("trim_messages() is not implemented yet - replace the TODO with your code")


history = [
    {"role": "system", "content": "You are Ada's twin."},
    {"role": "user", "content": "Hi there!"},
    {"role": "assistant", "content": "Hello! How can I help you today?"},
    {"role": "user", "content": "What do you cook?"},
]
# token sizes: 5, 3, 9, 5

assert trim_messages(history, 100) == history, "a budget of 100 should keep all 4 messages, got " + repr(trim_messages(history, 100))
assert trim_messages(history, 12) == [history[0], history[3]], "a budget of 12 should keep the system message and the newest turn, got " + repr(trim_messages(history, 12))
assert len(history) == 4, "trim_messages must not change the list it was given"
assert trim_messages(history, 1) == [history[0]], "a budget of 1 should keep only the system message, got " + repr(trim_messages(history, 1))

print("All checks passed.")


## Solution

In [ ]:
def count_tokens(text: str) -> int:
    return len(text) // 4 + 1


def trim_messages(messages: list[dict], max_tokens: int) -> list[dict]:
    system = messages[:1] if messages and messages[0]["role"] == "system" else []
    rest = messages[len(system):]
    used = sum(count_tokens(m["content"]) for m in system)
    kept = []
    for message in reversed(rest):
        cost = count_tokens(message["content"])
        if used + cost > max_tokens:
            break
        kept.append(message)
        used += cost
    kept.reverse()
    return system + kept


history = [
    {"role": "system", "content": "You are Ada's twin."},
    {"role": "user", "content": "Hi there!"},
    {"role": "assistant", "content": "Hello! How can I help you today?"},
    {"role": "user", "content": "What do you cook?"},
]
print([m["content"] for m in trim_messages(history, 12)])


## Real-world project (offline)

The code cell below builds a longer chat between a user and Ada's twin: a system message and 8 turns. Your project checks how big the whole history is, trims it to a small token budget with your `trim_messages`, and prints what survived. Nothing here calls an LLM. It is the step your code would run before every API call in a long chat.

In [ ]:
"""Load a longer chat history for the twin (embedded so this notebook needs no repo checkout)."""
# lab-data-loader
CHAT_HISTORY = [
    {"role": "system", "content": "You are Ada's digital twin. Answer as Ada, in a warm and short way."},
    {"role": "user", "content": "Hi! Who are you?"},
    {"role": "assistant", "content": "I am Ada's digital twin. Ask me about her work or her weekends."},
    {"role": "user", "content": "What does she do for work?"},
    {"role": "assistant", "content": "She writes Python code and builds web apps every day."},
    {"role": "user", "content": "And what does she do for fun?"},
    {"role": "assistant", "content": "She bakes bread, runs in the park and watches football."},
    {"role": "user", "content": "Has she travelled anywhere lately?"},
    {"role": "assistant", "content": "Last summer she flew to Accra and Lisbon."},
]

print(f"Loaded a chat history of {len(CHAT_HISTORY)} messages.")


In [ ]:
before = sum(count_tokens(m["content"]) for m in CHAT_HISTORY)
trimmed = trim_messages(CHAT_HISTORY, 60)
after = sum(count_tokens(m["content"]) for m in trimmed)

print(f"before: {len(CHAT_HISTORY)} messages, {before} tokens")
print(f"after: {len(trimmed)} messages, {after} tokens")
for message in trimmed:
    print(f"{message['role']}: {message['content']}")
